<a href="https://colab.research.google.com/github/israakadhem0-coder/Drug-discovery_ML/blob/main/Keap1_israa_code1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
!pip install chembl_webresource_client
from chembl_webresource_client.new_client import new_client

# 1. Target search for Keap1
target = new_client.target
target_query = target.search('Keap1')
targets = pd.DataFrame.from_dict(target_query)

# Display available targets to inspect organisms and ChEMBL IDs
print("Available Targets:")
print(targets[['target_chembl_id', 'pref_name', 'organism']].head())

# 2. Select the specific target (e.g., Human Keap1: CHEMBL1075138)
# Replace 'CHEMBL1075138' with your intended target ChEMBL ID if different
selected_target = 'CHEMBL1075138'

# 3. Retrieve bioactivity data for IC50
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)

# 4. Clean and filter dataset
# Keep only records with standard_units in nM to ensure consistent scale
df_filtered = df[df['standard_units'] == 'nM'].copy()

# Ensure standard_value is numeric and drop missing essential values
df_filtered['standard_value'] = pd.to_numeric(df_filtered['standard_value'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['standard_value', 'canonical_smiles', 'molecule_chembl_id'])

# 5. Classify bioactivity based on nM thresholds
# <= 1000 nM: Active | >= 10000 nM: Inactive | In-between: Intermediate
conditions = [
    df_filtered['standard_value'] <= 1000,
    df_filtered['standard_value'] >= 10000
]
choices = ['active', 'inactive']
df_filtered['bioactivity_class'] = np.select(conditions, choices, default='intermediate')

# 6. Extract clean dataframe with core columns
df_final = df_filtered[[
    'molecule_chembl_id',
    'canonical_smiles',
    'bioactivity_class',
    'standard_value'
]].reset_index(drop=True)

# Preview final dataset
print("\nProcessed Bioactivity Data:")
print(df_final.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.9 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


Available Targets:
  target_chembl_id                            pref_name           organism
0    CHEMBL3038498                           Keap1/Nrf2       Homo sapiens
1    CHEMBL4296095                           KEAP1/NRF2  Rattus norvegicus
2    CHEMBL4523596  Kelch-like ECH-associated protein 1  Rattus norvegicus
3    CHEMBL3562164  Kelch-like ECH-associated protein 1       Mus musculus
4    CHEMBL2069156  Kelch-like ECH-associated protein 1       Homo sapiens

Processed Bioactivity Data:
  molecule_chembl_id                                   canonical_smiles  \
0       CHEMBL387152  C[C@]12CC[C@H]3[C@@H](CCC4=CC(=O)CC[C@@]43C)[C...   
1       CHEMBL591237  CSc1ncnc2c1NCN2CC(=O)[C@H]1CC[C@H]2[C@@H]3CCC4...   
2       CHEMBL601298           Fc1cccc(Cl)c1/C=N/N=C1c2ccccc2-c2ccccc21   
3       CHEMBL591472  O=c1c2ccccc2c2ccccc2c(=O)n1-c1cccc2c1ccc1c3ccc...   
4       CHEMBL601283  CCOC(=O)CN(C(=O)COC(=O)Cc1c[nH]c2ccccc12)c1ccc...   

  bioactivity_class  standard_value  
0      interm